# argmax-accuracy-eval — worked example 2: Streaming accuracy meter over logit chunks

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `argmax-accuracy-eval`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

When evaluating in chunks you should accumulate **integer** counts `(correct, total)` rather than averaging per-chunk accuracies, because a partial final chunk must be weighted by its true size. Each chunk contributes `(preds == labels).sum()` correct out of `labels.shape[0]` total.

## Worked solution

**Step 1 — a tiny stateful meter.** We keep two Python ints, `correct` and `total`, both starting at 0. Integers avoid floating drift and make the final ratio exact.

**Step 2 — update per chunk.** For each `(logits_chunk, label_chunk)` we compute `preds = logits_chunk.argmax(dim=-1)`, then add `(preds == label_chunk).sum().item()` to `correct` and `label_chunk.shape[0]` to `total`. The `.item()` pulls the scalar count out as a Python int.

**Step 3 — final ratio.** `correct / total` is the exact overall accuracy. Because we summed raw counts, a chunk of size 2 contributes exactly 2 to `total`, not the same weight as a chunk of size 8.

**Why it works.** A weighted average of accuracies where each chunk is weighted by its size equals the global count-based ratio. Accumulating counts directly is the cleanest way to realize that weighting and is robust to ragged final chunks.

In [ ]:
class AccuracyMeter:
    def __init__(self):
        self.correct = 0
        self.total = 0
    def update(self, logits, labels):
        preds = logits.argmax(dim=-1)
        self.correct += (preds == labels).sum().item()
        self.total += labels.shape[0]
    def compute(self):
        return self.correct / self.total

t.manual_seed(0)
meter = AccuracyMeter()
for sz in [8, 8, 2]:          # last chunk is partial
    logits = t.randn(sz, 5)
    labels = t.randint(0, 5, (sz,))
    meter.update(logits, labels)
print(round(meter.compute(), 4))